# MSCOCO_CLIPS1: ONNX FP32 Export & Parity

Xuất encoder/decoder FP32, kiểm tra ONNX checker và parity PyTorch–ONNX Runtime.

## Input bắt buộc

1. Output MOB_MSCOCO_CLIPS1_Entry đã chạy lại bằng notebook đã sửa
2. Output MOB_MSCOCO_CLIPS1_ONNX_Preflight đã chạy lại bằng notebook đã sửa
3. Dataset ONNX_Runtime_Python313_Offline (chỉ cần khi runtime chưa cài sẵn)

Runtime: **CPU / Accelerator None / Internet Off**. Không huấn luyện, không chọn lại checkpoint và không đọc Test.


In [1]:

import csv
import hashlib
import json
import os
import shutil
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def unique_lock(suffix, model_id):
    matches = []
    for path in INPUT_ROOT.rglob(suffix):
        if path.is_file():
            payload = read_json(path)
            if payload.get("model_id") == model_id:
                matches.append((path, payload))
    assert len(matches) == 1, (suffix, model_id, [str(p) for p, _ in matches])
    return matches[0]

def file_by_hash(name, expected):
    matches = [p for p in INPUT_ROOT.rglob(name) if p.is_file() and sha256(p) == expected]
    assert len(matches) == 1, (name, expected, [str(p) for p in matches])
    return matches[0]

CFG = json.loads('{"short": "MSCOCO_CLIPS1", "teacher": "CLIP ViT-B/32", "student": "MobileNetV3-Large", "student_id": "S1", "dataset": "MS COCO 2014", "method": "Direct Logit-KD", "seed": 123, "vocab_name": "word_vocabulary_minfreq5.json", "vocab_key": "id_to_token", "vocab_size": 10080, "encoder_key": "student_encoder_state", "decoder_key": "student_decoder_state", "source_dim": 960, "encoder_params": 3231024, "decoder_params": 5759328, "decoder_batch": 3, "mask_dtype": "int64", "encoder_input": "image", "encoder_output": "visual_memory", "token_input": "decoder_input_ids", "memory_input": "visual_memory", "mask_input": "decoder_attention_mask", "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225], "validation_file": "mscoco_validation.parquet"}')
EXPERIMENT_ID = "MOB_" + CFG["short"] + "_ONNX_FP32"
OUT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
EXPORTS, PACKAGE, AUDITS, RESULTS = [OUT / name for name in ("exports", "package", "audits", "results")]
for directory in (EXPORTS, PACKAGE, AUDITS, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)

entry_path, entry = unique_lock("*_ENTRY_LOCK.json", CFG["short"])
preflight_path, preflight = unique_lock("*_ONNX_PREFLIGHT.json", CFG["short"])
assert entry["status"] == "locked" and preflight["status"] == "passed"
assert entry["selected_seed"] == CFG["seed"] == preflight["selected_seed"]
assert entry["test_set_used"] is False and preflight["test_set_used"] is False
checkpoint = file_by_hash("selected_best.pt", entry["checkpoint"]["sha256"])
vocabulary = file_by_hash(CFG["vocab_name"], entry["vocabulary"]["sha256"])
vocab_payload = read_json(vocabulary)
assert len(vocab_payload[CFG["vocab_key"]]) == CFG["vocab_size"]

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_grad_enabled(False)
torch.manual_seed(int(CFG["seed"]))
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

def select_tensor(value):
    if torch.is_tensor(value):
        return value
    if isinstance(value, dict):
        return [item for item in value.values() if torch.is_tensor(item)][-1]
    if isinstance(value, (tuple, list)):
        return [item for item in value if torch.is_tensor(item)][-1]
    raise TypeError(type(value))

class VisualEncoder(nn.Module):
    def __init__(self, student_id, source_dim):
        super().__init__()
        self.student_id = student_id
        if student_id == "S1":
            from torchvision.models import mobilenet_v3_large
            self.backbone = mobilenet_v3_large(weights=None).features
        elif student_id == "S2":
            import timm
            self.backbone = timm.create_model("mobilevit_s.cvnets_in1k", pretrained=False)
            self.backbone.reset_classifier(num_classes=0, global_pool="")
            for name in ("head", "head_dist", "classifier", "fc"):
                head = getattr(self.backbone, name, None)
                if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
                    setattr(self.backbone, name, nn.Identity())
        else:
            raise ValueError(student_id)
        self.source_dim = source_dim
        self.projection = nn.Linear(source_dim, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        if self.student_id == "S1":
            feature = self.backbone(images)
        else:
            feature = select_tensor(self.backbone.forward_features(images))
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(self.projection(tokens) + self.visual_position_embedding)

class CaptionDecoder(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.d_model = 256
        self.maximum_steps = 31
        self.token_embedding = nn.Embedding(vocabulary_size, 256, padding_idx=0)
        self.position_embedding = nn.Embedding(31, 256)
        self.embedding_dropout = nn.Dropout(0.1)
        layer = nn.TransformerDecoderLayer(
            256, 8, 1024, 0.1, "gelu", batch_first=True, norm_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(layer, 3)
        self.final_norm = nn.LayerNorm(256)
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))

    def forward(self, token_ids, memory, attention_mask):
        positions = torch.arange(31, device=token_ids.device).unsqueeze(0).expand(token_ids.shape[0], -1)
        hidden = self.embedding_dropout(
            self.token_embedding(token_ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        causal = torch.triu(
            torch.ones(31, 31, dtype=torch.bool, device=token_ids.device), diagonal=1
        )
        hidden = self.transformer_decoder(
            hidden,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~attention_mask.bool(),
            tgt_is_causal=True,
        )
        return F.linear(self.final_norm(hidden), self.token_embedding.weight, self.output_bias)

class ExportSafeDecoder(nn.Module):
    def __init__(self, source):
        super().__init__()
        self.d_model = source.d_model
        self.maximum_steps = source.maximum_steps
        self.token_embedding = source.token_embedding
        self.position_embedding = source.position_embedding
        self.layers = source.transformer_decoder.layers
        self.final_norm = source.final_norm
        self.output_bias = source.output_bias
        self.register_buffer(
            "causal_mask",
            torch.triu(torch.ones(31, 31, dtype=torch.bool), diagonal=1),
            persistent=False,
        )

    def _attention(self, module, query, key, value, valid_keys, causal):
        width = self.d_model
        heads = module.num_heads
        head_width = width // heads
        weight, bias = module.in_proj_weight, module.in_proj_bias
        q = F.linear(query, weight[:width], bias[:width])
        k = F.linear(key, weight[width:2 * width], bias[width:2 * width])
        v = F.linear(value, weight[2 * width:], bias[2 * width:])
        batch, query_length, key_length = query.shape[0], query.shape[1], key.shape[1]
        q = q.reshape(batch, query_length, heads, head_width).transpose(1, 2)
        k = k.reshape(batch, key_length, heads, head_width).transpose(1, 2)
        v = v.reshape(batch, key_length, heads, head_width).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_width)
        if causal:
            scores = scores.masked_fill(
                self.causal_mask[:query_length, :key_length].unsqueeze(0).unsqueeze(0), -1.0e4
            )
        if valid_keys is not None:
            scores = scores.masked_fill(~valid_keys.bool().unsqueeze(1).unsqueeze(1), -1.0e4)
        probabilities = torch.softmax(scores, dim=-1)
        context = torch.matmul(probabilities, v)
        context = context.transpose(1, 2).contiguous().reshape(batch, query_length, width)
        return F.linear(context, module.out_proj.weight, module.out_proj.bias)

    def forward(self, token_ids, memory, attention_mask):
        positions = torch.arange(31, device=token_ids.device).unsqueeze(0).expand(token_ids.shape[0], -1)
        hidden = self.token_embedding(token_ids) * math.sqrt(self.d_model) + self.position_embedding(positions)
        for layer in self.layers:
            normalized = layer.norm1(hidden)
            hidden = hidden + self._attention(
                layer.self_attn, normalized, normalized, normalized, attention_mask, True
            )
            normalized = layer.norm2(hidden)
            hidden = hidden + self._attention(
                layer.multihead_attn, normalized, memory, memory, None, False
            )
            normalized = layer.norm3(hidden)
            hidden = hidden + F.linear(
                F.gelu(F.linear(normalized, layer.linear1.weight, layer.linear1.bias)),
                layer.linear2.weight,
                layer.linear2.bias,
            )
        return F.linear(self.final_norm(hidden), self.token_embedding.weight, self.output_bias)

encoder = VisualEncoder(CFG["student_id"], int(CFG["source_dim"])).cpu().eval()
decoder = CaptionDecoder(int(CFG["vocab_size"])).cpu().eval()
payload = torch.load(checkpoint, map_location="cpu", weights_only=False)
encoder_result = encoder.load_state_dict(payload[CFG["encoder_key"]], strict=True)
decoder_result = decoder.load_state_dict(payload[CFG["decoder_key"]], strict=True)
assert not encoder_result.missing_keys and not encoder_result.unexpected_keys
assert not decoder_result.missing_keys and not decoder_result.unexpected_keys
assert sum(p.numel() for p in encoder.parameters()) == CFG["encoder_params"]
assert sum(p.numel() for p in decoder.parameters()) == CFG["decoder_params"]

batch = int(CFG["decoder_batch"])
image = torch.linspace(-2.0, 2.0, 1 * 3 * 224 * 224).reshape(1, 3, 224, 224)
token_ids = torch.zeros((batch, 31), dtype=torch.long)
if CFG["mask_dtype"] == "int64":
    attention_mask = torch.zeros((batch, 31), dtype=torch.int64)
else:
    attention_mask = torch.zeros((batch, 31), dtype=torch.bool)
for row in range(batch):
    length = min(31, 1 + row * 10)
    token_ids[row, 0] = 1
    if length > 1:
        token_ids[row, 1:length] = torch.arange(4, 4 + length - 1)
    attention_mask[row, :length] = 1

with torch.inference_mode():
    memory = encoder(image)
    decoder_memory = memory.expand(batch, -1, -1).contiguous()
    reference_logits = decoder(token_ids, decoder_memory, attention_mask)
    export_decoder = ExportSafeDecoder(decoder).cpu().eval()
    export_logits = export_decoder(token_ids, decoder_memory, attention_mask)
wrapper_error = float((reference_logits - export_logits).abs().max().item())
assert wrapper_error <= 2e-4, wrapper_error
assert torch.equal(reference_logits.argmax(-1), export_logits.argmax(-1))

try:
    import onnxruntime as ort
except ImportError:
    wheel_dirs = sorted({p.parent for p in INPUT_ROOT.rglob("onnxruntime-*.whl")})
    assert wheel_dirs, "Gắn dataset ONNX_Runtime_Python313_Offline"
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-index",
        "--find-links", str(wheel_dirs[0]), "onnxruntime",
    ])
    import onnxruntime as ort
import onnx

encoder_onnx = EXPORTS / (CFG["short"].lower() + "_encoder_fp32.onnx")
decoder_onnx = EXPORTS / (CFG["short"].lower() + "_decoder_fp32.onnx")
torch.onnx.export(
    encoder, (image,), str(encoder_onnx),
    input_names=[CFG["encoder_input"]], output_names=[CFG["encoder_output"]],
    opset_version=18, export_params=True, do_constant_folding=True, dynamo=False,
)
torch.onnx.export(
    export_decoder, (token_ids, decoder_memory, attention_mask), str(decoder_onnx),
    input_names=[CFG["token_input"], CFG["memory_input"], CFG["mask_input"]],
    output_names=["logits"], opset_version=18, export_params=True,
    do_constant_folding=True, dynamo=False,
)
for path in (encoder_onnx, decoder_onnx):
    graph = onnx.load(str(path), load_external_data=True)
    onnx.checker.check_model(graph, full_check=True)
    assert path.stat().st_size > 1_000_000

encoder_session = ort.InferenceSession(str(encoder_onnx), providers=["CPUExecutionProvider"])
decoder_session = ort.InferenceSession(str(decoder_onnx), providers=["CPUExecutionProvider"])
cases = []
for case_seed in (42, 123, 2026):
    generator = torch.Generator().manual_seed(case_seed)
    case_image = torch.randn((1, 3, 224, 224), generator=generator)
    with torch.inference_mode():
        pt_memory = encoder(case_image)
        pt_decoder_memory = pt_memory.expand(batch, -1, -1).contiguous()
        pt_logits = export_decoder(token_ids, pt_decoder_memory, attention_mask)
    ort_memory = encoder_session.run(None, {CFG["encoder_input"]: case_image.numpy()})[0]
    ort_decoder_memory = np.repeat(ort_memory, batch, axis=0)
    ort_logits = decoder_session.run(None, {
        CFG["token_input"]: token_ids.numpy(),
        CFG["memory_input"]: ort_decoder_memory,
        CFG["mask_input"]: attention_mask.numpy(),
    })[0]
    cases.append({
        "seed": case_seed,
        "encoder_max_abs_error": float(np.max(np.abs(pt_memory.numpy() - ort_memory))),
        "decoder_max_abs_error": float(np.max(np.abs(pt_logits.numpy() - ort_logits))),
        "top1_agreement": float(np.mean(pt_logits.numpy().argmax(-1) == ort_logits.argmax(-1))),
    })
assert max(row["encoder_max_abs_error"] for row in cases) <= 1e-4
assert max(row["decoder_max_abs_error"] for row in cases) <= 2e-3
assert min(row["top1_agreement"] for row in cases) == 1.0

package_encoder = PACKAGE / encoder_onnx.name
package_decoder = PACKAGE / decoder_onnx.name
package_vocab = PACKAGE / CFG["vocab_name"]
shutil.copy2(encoder_onnx, package_encoder)
shutil.copy2(decoder_onnx, package_decoder)
shutil.copy2(vocabulary, package_vocab)
contract = {
    "schema_version": "mobile_fp32_contract_v1",
    "created_at_utc": now(),
    "status": "complete",
    "model_id": CFG["short"],
    "teacher": CFG["teacher"],
    "student": CFG["student"],
    "dataset": CFG["dataset"],
    "method": CFG["method"],
    "selected_seed": CFG["seed"],
    "precision": "FP32",
    "preprocessing": {"mean": CFG["mean"], "std": CFG["std"], "image": [1, 3, 224, 224]},
    "encoder": {
        "file": package_encoder.name, "input_name": CFG["encoder_input"],
        "output_name": CFG["encoder_output"], "output_shape": [1, 49, 256],
        "bytes": package_encoder.stat().st_size, "sha256": sha256(package_encoder),
    },
    "decoder": {
        "file": package_decoder.name, "batch": batch, "steps": 31,
        "vocabulary_size": CFG["vocab_size"], "mask_dtype": CFG["mask_dtype"],
        "input_names": [CFG["token_input"], CFG["memory_input"], CFG["mask_input"]],
        "bytes": package_decoder.stat().st_size, "sha256": sha256(package_decoder),
    },
    "vocabulary": {"file": package_vocab.name, "sha256": sha256(package_vocab)},
    "checkpoint_sha256": sha256(checkpoint),
    "wrapper_equivalence_max_abs_error": wrapper_error,
    "parity_cases": cases,
    "selection_split": "validation",
    "test_set_used": False,
    "authorized_next": "INT8_Preflight_after_Calib_Lock",
}
contract_path = OUT / (EXPERIMENT_ID + "_COMPLETE.json")
write_json(contract_path, contract)

artifacts = [package_encoder, package_decoder, package_vocab, contract_path]
with (OUT / "MANIFEST.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["relative_path", "bytes", "sha256"])
    writer.writeheader()
    for path in artifacts:
        writer.writerow({
            "relative_path": path.relative_to(OUT).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256(path),
        })

print("ONNX FP32 export        : PASSED")
print("Model                   :", CFG["teacher"], "->", CFG["student"])
print("Dataset / KD            :", CFG["dataset"], "/", CFG["method"])
print("Selected seed           :", CFG["seed"])
print("Encoder / decoder MiB   :", round(package_encoder.stat().st_size / 2**20, 2), "/", round(package_decoder.stat().st_size / 2**20, 2))
print("PyTorch-ORT parity      : PASSED")
print("Test                    : SEALED / NOT READ")


Looking in links: /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline
Processing /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline/onnxruntime-1.28.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


/tmp/ipykernel_17/3499531220.py:255: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_17/3499531220.py:260: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX FP32 export        : PASSED
Model                   : CLIP ViT-B/32 -> MobileNetV3-Large
Dataset / KD            : MS COCO 2014 / Direct Logit-KD
Selected seed           : 123
Encoder / decoder MiB   : 12.32 / 31.84
PyTorch-ORT parity      : PASSED
Test                    : SEALED / NOT READ
